# Data Cleaning and Standardization

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/Bacteria_dataset_Multiresictance.csv")

In [3]:
df_clean = df.copy()

In [4]:
for column in df_clean.select_dtypes(include="object"):
    print(f"\n{column}")
    print("-" * len(column))
    print(df_clean[column].unique())


ID
--
['S290' 'S291' 'S292' ... 'S10997' 'S10998' 'S10999']

Name
----
['Elizabeth Lawrence' 'Tina Sanders' 'Erin Cooke' ... 'Alexandra Smith'
 'April Cox' 'Richard Jarvis']

Email
-----
['elizabeth.lawrence@example.com' 'tina.sanders@example.com'
 'erin.cooke@example.com' ... 'alexandra.smith@example.com'
 'april.cox@example.com' 'richard.jarvis@example.com']

Address
-------
['6350 Robinson Loaf Apt. 447, Paulfurt, RI 30252'
 '78594 Galloway Port Suite 762, South Tanyatown, HI 30310'
 '76661 Isaiah Manors, North Benjamin, HI 85195' ...
 '213 Yates Station, Port Meganstad, PW 17799'
 '985 Julia Freeway Apt. 753, Dianamouth, SD 07610'
 '2903 James Motorway Suite 202, Whitakerside, CT 62052']

age/gender
----------
['37/F' '29/F' '77/F' nan '13/F' '57/M' '68/F' '18/F' '36/F' '49/M' '47/F'
 '74/M' '27/F' '34/F' '19/F' '57/F' '38/F' '71/F' '54/M' '73/F' '66/F'
 '50/F' '52/M' '88/M' '80/F' '16/F' '47/M' '86/F' '17/M' '76/F' '79/F'
 '20/M' '28/F' '51/F' '35/F' '74/F' '2/F' '58/F' '22/F' '3

## Fixing missing value placeholders

In [5]:
missing_values = [
    "missing",
    "?",
    "unknown",
    "error"
]

df_clean.replace(missing_values, np.nan, inplace=True)
for value in missing_values:
    print(f"{value}: {(df_clean == value).sum().sum()}")
df_clean.isna().sum()

missing: 0
?: 0
unknown: 0
error: 0


ID                      0
Name                    0
Email                   0
Address                 0
age/gender            763
Souches               763
Diabetes              763
Hypertension          763
Hospital_before       763
Infection_Freq        996
AMX/AMP               753
AMC                   753
CZ                    753
FOX                   753
CTX/CRO               753
IPM                   753
GEN                   753
AN                    753
Acide nalidixique     753
ofx                   753
CIP                   753
C                     753
Co-trimoxazole        753
Furanes               753
colistine             753
Collection_Date      2315
Notes                 244
dtype: int64

## Decouple the combined 'age/gender' column

In [6]:
df_clean[["Age", "Gender"]] = df_clean["age/gender"].str.split("/", expand=True)

df_clean["Age"] = pd.to_numeric(df_clean["Age"], errors="coerce")
df_clean["Age"] = df_clean["Age"].astype("Int64")
df_clean["Gender"] = df_clean["Gender"].str.upper()

In [7]:
df_clean["Gender"].value_counts(dropna=False)

Gender
F      7929
M      2018
NaN     763
Name: count, dtype: int64

In [8]:
df_clean["Age"].describe()

count       9947.0
mean     45.601789
std      24.891645
min            0.0
25%           25.0
50%           45.0
75%           67.0
max           90.0
Name: Age, dtype: Float64

In [9]:
df_clean.drop(columns="age/gender", inplace=True)

## Encoding binary columns

In [10]:
binary_columns = ["Gender", "Diabetes", "Hypertension", "Hospital_before"]
for col in binary_columns:
    print(df_clean[col].value_counts(dropna=False))

Gender
F      7929
M      2018
NaN     763
Name: count, dtype: int64
Diabetes
No      7883
True    2064
NaN      763
Name: count, dtype: int64
Hypertension
No     7470
Yes    2477
NaN     763
Name: count, dtype: int64
Hospital_before
No     7008
Yes    2939
NaN     763
Name: count, dtype: int64


In [11]:
# Binary encoding mapping
binary_mapping = {
    "F": 0,
    "M": 1,
    "No": 0,
    "Yes": 1,
    "True": 1
}
for col in binary_columns:
    df_clean[col] = df_clean[col].replace(binary_mapping).astype("Int64")

C:\Users\Hridesh\AppData\Local\Temp\ipykernel_25788\3591385858.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean[col] = df_clean[col].replace(binary_mapping).astype("Int64")
C:\Users\Hridesh\AppData\Local\Temp\ipykernel_25788\3591385858.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean[col] = df_clean[col].replace(binary_mapping).astype("Int64")
C:\Users\Hridesh\AppData\Local\Temp\ipykernel_25788\3591385858.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future

In [12]:
for col in binary_columns:
    print(df_clean[col].value_counts(dropna=False))

Gender
0       7929
1       2018
<NA>     763
Name: count, dtype: Int64
Diabetes
0       7883
1       2064
<NA>     763
Name: count, dtype: Int64
Hypertension
0       7470
1       2477
<NA>     763
Name: count, dtype: Int64
Hospital_before
0       7008
1       2939
<NA>     763
Name: count, dtype: Int64


## Pre-processing Infection_Freq colmn

In [13]:
df_clean["Infection_Freq"].value_counts(dropna=False)

Infection_Freq
2.0    2966
1.0    2889
3.0    1963
0.0    1896
NaN     996
Name: count, dtype: int64

In [14]:
df_clean["Infection_Freq"] = pd.to_numeric(
    df_clean["Infection_Freq"],
    errors="coerce"
)

df_clean["Infection_Freq"] = df_clean["Infection_Freq"].astype("Int64")

In [15]:
df_clean["Infection_Freq"].dtype

Int64Dtype()

## Pre-processing Anti-Biotics colmn

In [16]:
antibiotic_columns = [
    "AMX/AMP",
    "AMC",
    "CZ",
    "FOX",
    "CTX/CRO",
    "IPM",
    "GEN",
    "AN",
    "Acide nalidixique",
    "ofx",
    "CIP",
    "C",
    "Co-trimoxazole",
    "Furanes",
    "colistine"
]

for col in antibiotic_columns:
    print(df_clean[col].value_counts(dropna=False))

AMX/AMP
R               5634
S               3961
NaN              753
s                100
i                 93
Intermediate      91
r                 78
Name: count, dtype: int64
AMC
R               5719
S               3876
NaN              753
i                 97
r                 96
s                 94
Intermediate      75
Name: count, dtype: int64
CZ
R               5601
S               3994
NaN              753
s                103
Intermediate      98
r                 89
i                 72
Name: count, dtype: int64
FOX
R               5644
S               3951
NaN              753
s                 98
i                 93
r                 91
Intermediate      80
Name: count, dtype: int64
CTX/CRO
R               5660
S               3935
NaN              753
i                 95
s                 91
r                 88
Intermediate      88
Name: count, dtype: int64
IPM
R               5635
S               3960
NaN              753
i                 94
s                 94

In [17]:
# Antibiotic susceptibility encoding
antibiotic_mapping = {
    "S": 0,
    "s": 0,
    "Intermediate": 1,
    "i": 1,
    "R": 2,
    "r": 2
}

for col in antibiotic_columns:
    df_clean[col] = (
        df_clean[col]
        .replace(antibiotic_mapping)
        .astype("Int64")
    )
for col in antibiotic_columns:
    print(df_clean[col].value_counts(dropna=False))

C:\Users\Hridesh\AppData\Local\Temp\ipykernel_25788\1301181372.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(antibiotic_mapping)
C:\Users\Hridesh\AppData\Local\Temp\ipykernel_25788\1301181372.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(antibiotic_mapping)
C:\Users\Hridesh\AppData\Local\Temp\ipykernel_25788\1301181372.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=F

AMX/AMP
2       5712
0       4061
<NA>     753
1        184
Name: count, dtype: Int64
AMC
2       5815
0       3970
<NA>     753
1        172
Name: count, dtype: Int64
CZ
2       5690
0       4097
<NA>     753
1        170
Name: count, dtype: Int64
FOX
2       5735
0       4049
<NA>     753
1        173
Name: count, dtype: Int64
CTX/CRO
2       5748
0       4026
<NA>     753
1        183
Name: count, dtype: Int64
IPM
2       5717
0       4054
<NA>     753
1        186
Name: count, dtype: Int64
GEN
0       7854
2       1936
<NA>     753
1        167
Name: count, dtype: Int64
AN
0       7875
2       1905
<NA>     753
1        177
Name: count, dtype: Int64
Acide nalidixique
0       8379
2       1388
<NA>     753
1        190
Name: count, dtype: Int64
ofx
0       8406
2       1381
<NA>     753
1        170
Name: count, dtype: Int64
CIP
0       8331
2       1447
<NA>     753
1        179
Name: count, dtype: Int64
C
0       8390
2       1385
<NA>     753
1        182
Name: count, dtype: Int6

## Extracting the bacterial species from the Souches column

In [18]:
print(df_clean["Souches"].value_counts(dropna=False))

Souches
NaN                           763
S4777 Escherichia coli          3
S597 Escherichia coli           3
S2424 Prot.eus mirabilis        3
S3413 Escherichia coli          3
                             ... 
S10269 Enterobacteria spp.      1
S10270 Escherichia coli         1
S10271 Enter.bacteria spp.      1
S10272 Escherichia coli         1
S10258 Proteus mirabilis        1
Name: count, Length: 9297, dtype: int64


In [19]:
df_clean["Bacteria"] = df_clean["Souches"].str.split(" ", n=1).str[1]
sorted(df_clean["Bacteria"].dropna().unique())

['Acinetobacter baumannii',
 'Citrobacter spp.',
 'E. coli',
 'E.cli',
 'E.coi',
 'Enteobacteria spp.',
 'Enter.bacteria spp.',
 'Enterobacteria spp.',
 'Escherichia coli',
 'Klbsiella pneumoniae',
 'Klebsie.lla pneumoniae',
 'Klebsiella pneumoniae',
 'Morganella morganii',
 'Proeus mirabilis',
 'Prot.eus mirabilis',
 'Proteus mirabilis',
 'Protus mirabilis',
 'Pseudomonas aeruginosa',
 'Serratia marcescens']

In [20]:
species_mapping = {
    # Escherichia coli
    "Escherichia coli": "Escherichia coli",
    "E. coli": "Escherichia coli",
    "E.cli": "Escherichia coli",
    "E.coi": "Escherichia coli",

    # Klebsiella pneumoniae
    "Klebsiella pneumoniae": "Klebsiella pneumoniae",
    "Klbsiella pneumoniae": "Klebsiella pneumoniae",
    "Klebsie.lla pneumoniae": "Klebsiella pneumoniae",

    # Proteus mirabilis
    "Proteus mirabilis": "Proteus mirabilis",
    "Protus mirabilis": "Proteus mirabilis",
    "Proeus mirabilis": "Proteus mirabilis",
    "Prot.eus mirabilis": "Proteus mirabilis",

    # Enterobacteria spp.
    "Enterobacteria spp.": "Enterobacteria spp.",
    "Enter.bacteria spp.": "Enterobacteria spp.",
    "Enteobacteria spp.": "Enterobacteria spp.",

    # Others
    "Morganella morganii": "Morganella morganii",
    "Citrobacter spp.": "Citrobacter spp.",
    "Pseudomonas aeruginosa": "Pseudomonas aeruginosa",
    "Serratia marcescens": "Serratia marcescens",
    "Acinetobacter baumannii": "Acinetobacter baumannii",
}
df_clean["Bacteria"] = df_clean["Bacteria"].replace(species_mapping)
df_clean["Bacteria"].value_counts(dropna=False)

Bacteria
Escherichia coli           6083
Enterobacteria spp.         997
NaN                         763
Proteus mirabilis           742
Klebsiella pneumoniae       702
Citrobacter spp.            481
Morganella morganii         305
Serratia marcescens         256
Pseudomonas aeruginosa      200
Acinetobacter baumannii     181
Name: count, dtype: int64

In [21]:
df_clean.drop(columns="Souches", inplace=True)

## Removing identifier and metadata columns

In [22]:
print(df_clean.columns.tolist())

['ID', 'Name', 'Email', 'Address', 'Diabetes', 'Hypertension', 'Hospital_before', 'Infection_Freq', 'AMX/AMP', 'AMC', 'CZ', 'FOX', 'CTX/CRO', 'IPM', 'GEN', 'AN', 'Acide nalidixique', 'ofx', 'CIP', 'C', 'Co-trimoxazole', 'Furanes', 'colistine', 'Collection_Date', 'Notes', 'Age', 'Gender', 'Bacteria']


In [23]:
columns_to_drop = [
    "ID",
    "Name",
    "Email",
    "Address",
    "Collection_Date",
    "Notes"
]

df_clean.drop(columns=columns_to_drop, errors="ignore", inplace=True)

In [24]:
print(df_clean.shape)

(10710, 22)


In [25]:
df_clean.isna().sum()

Diabetes             763
Hypertension         763
Hospital_before      763
Infection_Freq       996
AMX/AMP              753
AMC                  753
CZ                   753
FOX                  753
CTX/CRO              753
IPM                  753
GEN                  753
AN                   753
Acide nalidixique    753
ofx                  753
CIP                  753
C                    753
Co-trimoxazole       753
Furanes              753
colistine            753
Age                  763
Gender               763
Bacteria             763
dtype: int64

In [26]:
#dataset health check

print("Shape:", df_clean.shape)

print("\nData Types:")
print(df_clean.dtypes)

print("\nMissing Values:")
print(df_clean.isna().sum())

print("\nPreview:")
df_clean.head()

Shape: (10710, 22)

Data Types:
Diabetes              Int64
Hypertension          Int64
Hospital_before       Int64
Infection_Freq        Int64
AMX/AMP               Int64
AMC                   Int64
CZ                    Int64
FOX                   Int64
CTX/CRO               Int64
IPM                   Int64
GEN                   Int64
AN                    Int64
Acide nalidixique     Int64
ofx                   Int64
CIP                   Int64
C                     Int64
Co-trimoxazole        Int64
Furanes               Int64
colistine             Int64
Age                   Int64
Gender                Int64
Bacteria             object
dtype: object

Missing Values:
Diabetes             763
Hypertension         763
Hospital_before      763
Infection_Freq       996
AMX/AMP              753
AMC                  753
CZ                   753
FOX                  753
CTX/CRO              753
IPM                  753
GEN                  753
AN                   753
Acide nalidixique    

,Diabetes,Hypertension,Hospital_before,Infection_Freq,AMX/AMP,AMC,CZ,FOX,CTX/CRO,IPM,...,Acide nalidixique,ofx,CIP,C,Co-trimoxazole,Furanes,colistine,Age,Gender,Bacteria
0,0,0,0,0,2,2,2,2,2,2,...,0,0,0,2,0,0,0,37,0,Escherichia coli
1,1,0,0,3,0,2,0,2,0,2,...,0,0,0,0,0,0,0,29,0,Morganella morganii
2,1,0,0,3,0,2,0,2,0,0,...,0,0,2,2,0,0,0,77,0,Proteus mirabilis
3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN
4,0,1,0,1,0,0,2,2,2,2,...,0,0,0,0,0,0,0,13,0,Escherichia coli


In [27]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_dir / "Bacteria_dataset_cleaned.csv", index=False)